<a href="https://colab.research.google.com/github/Anushadhirde/Urban-Heat-Island-Change-Detection/blob/main/Tile_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
"""
STEP: Tile the stacked input rasters and ground truth masks into small
matching patches, ready for model training.

WHAT THIS DOES:
- Opens each year/season's 8-band stacked raster and its matching mask
- Drops band 1 (LST) from the input, since LST was used to build the
  ground truth labels — keeping it as input would let the model "cheat"
  by just re-deriving the threshold instead of learning real patterns.
  Input tiles end up with 7 bands: NDVI, NDBI, NDWI, Albedo, Elevation,
  Slope, Aspect.
- Cuts both the input stack and its mask into TILE_SIZE x TILE_SIZE
  patches, using the SAME pixel boundaries for both, so tile #42's input
  always matches tile #42's mask.
- Skips any tile where too much of it is "no data" (outside your AOI),
  since those tiles have nothing for the model to learn from.
- Saves each kept tile as a pair of .npy files (one input, one mask) plus
  a manifest CSV listing every tile that was created, for traceability.

WHY .npy AND NOT .tif FOR THE TILES:
.npy is just a saved numpy array — smaller, faster to load in bulk during
training, and this is what most deep learning training loops expect. We
already have the full georeferenced .tif files as the source of truth if
you ever need to trace a tile back to its real-world location.

BEFORE YOU RUN:
- Confirm the paths below match your setup.
- TILE_SIZE defaults to 224 (a common input size for vision models,
  including Prithvi) — change it if your model expects something else.
"""

import os
import csv
import rasterio
import numpy as np

STACKED_FOLDER = "/content/drive/MyDrive/DATASET/STACKED"
MASKS_FOLDER = "/content/drive/MyDrive/DATASET/MASKS"
OUTPUT_FOLDER = "/content/drive/MyDrive/DATASET/TILES"

YEARS = list(range(2017, 2026))
SEASONS = ["Summer", "Winter"]

TILE_SIZE = 224          # width/height of each tile in pixels
MAX_NODATA_FRACTION = 0.5  # skip a tile if more than 50% of it is nodata

LST_BAND_INDEX = 1        # band 1 in the stacked files = LST (to be dropped)
MASK_NODATA_VALUE = 255   # matches NODATA_CLASS from the mask-generation script


def tile_one_scene(year, season, manifest_rows, tile_counter):
    stack_path = os.path.join(STACKED_FOLDER, f"stack_{year}_{season}.tif")
    mask_path = os.path.join(MASKS_FOLDER, f"mask_{year}_{season}.tif")

    if not os.path.exists(stack_path) or not os.path.exists(mask_path):
        print(f"  MISSING stack or mask for {year} {season} — skipping")
        return tile_counter

    with rasterio.open(stack_path) as src:
        all_bands = src.read()  # shape: (8, height, width)
        stack_nodata = src.nodata
        height, width = src.height, src.width

    # drop LST (band 1) -> keep bands 2 through 8 (index 1 onward, 0-indexed)
    input_bands = all_bands[LST_BAND_INDEX:, :, :]  # shape: (7, height, width)

    with rasterio.open(mask_path) as src:
        mask = src.read(1)  # shape: (height, width)

    kept, skipped = 0, 0

    for row_start in range(0, height, TILE_SIZE):
        for col_start in range(0, width, TILE_SIZE):
            row_end = row_start + TILE_SIZE
            col_end = col_start + TILE_SIZE

            # skip incomplete tiles at the edges (smaller than TILE_SIZE)
            if row_end > height or col_end > width:
                continue

            mask_tile = mask[row_start:row_end, col_start:col_end]
            nodata_fraction = np.mean(mask_tile == MASK_NODATA_VALUE)

            if nodata_fraction > MAX_NODATA_FRACTION:
                skipped += 1
                continue

            input_tile = input_bands[:, row_start:row_end, col_start:col_end]

            tile_counter += 1
            tile_id = f"tile_{tile_counter:05d}"

            np.save(os.path.join(OUTPUT_FOLDER, "inputs", f"{tile_id}.npy"), input_tile)
            np.save(os.path.join(OUTPUT_FOLDER, "masks", f"{tile_id}.npy"), mask_tile)

            manifest_rows.append({
                "tile_id": tile_id,
                "year": year,
                "season": season,
                "row_start": row_start,
                "col_start": col_start,
                "nodata_fraction": round(float(nodata_fraction), 4),
            })
            kept += 1

    print(f"{year} {season}: kept {kept} tiles, skipped {skipped} (too much nodata)")
    return tile_counter


def main():
    os.makedirs(os.path.join(OUTPUT_FOLDER, "inputs"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_FOLDER, "masks"), exist_ok=True)

    manifest_rows = []
    tile_counter = 0

    for year in YEARS:
        for season in SEASONS:
            tile_counter = tile_one_scene(year, season, manifest_rows, tile_counter)

    manifest_path = os.path.join(OUTPUT_FOLDER, "tiles_manifest.csv")
    if manifest_rows:
        fieldnames = list(manifest_rows[0].keys())
        with open(manifest_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(manifest_rows)

    print(f"\nDone. {len(manifest_rows)} tiles saved to {OUTPUT_FOLDER}")
    print(f"Manifest saved to {manifest_path}")
    print("Each input tile has 7 bands: NDVI, NDBI, NDWI, Albedo, Elevation, Slope, Aspect")
    print("(LST was excluded from inputs since it was used to build the ground truth masks)")


if __name__ == "__main__":
    main()

2017 Summer: kept 219 tiles, skipped 199 (too much nodata)
2017 Winter: kept 219 tiles, skipped 199 (too much nodata)
2018 Summer: kept 219 tiles, skipped 199 (too much nodata)
2018 Winter: kept 219 tiles, skipped 199 (too much nodata)
2019 Summer: kept 219 tiles, skipped 199 (too much nodata)
2019 Winter: kept 219 tiles, skipped 199 (too much nodata)
2020 Summer: kept 219 tiles, skipped 199 (too much nodata)
2020 Winter: kept 219 tiles, skipped 199 (too much nodata)
2021 Summer: kept 219 tiles, skipped 199 (too much nodata)
2021 Winter: kept 219 tiles, skipped 199 (too much nodata)
2022 Summer: kept 219 tiles, skipped 199 (too much nodata)
2022 Winter: kept 219 tiles, skipped 199 (too much nodata)
2023 Summer: kept 219 tiles, skipped 199 (too much nodata)
2023 Winter: kept 219 tiles, skipped 199 (too much nodata)
2024 Summer: kept 219 tiles, skipped 199 (too much nodata)
2024 Winter: kept 219 tiles, skipped 199 (too much nodata)
2025 Summer: kept 219 tiles, skipped 199 (too much nodat